In [ ]:
import os, numpy as np, pandas as pd, anndata, matplotlib.pyplot as plt, dynamo as dyn

# ═══════════════ 1. READ NEG-CONTROL CSVs ══════════════════════════════════
data_dir, dims = "./data/neg_control", [2, 8, 32, 128]
datasets = {
    f"d{d}": {
        "X": pd.read_csv(os.path.join(data_dir, f"X_d{d}.csv")).values,
        "V": pd.read_csv(os.path.join(data_dir, f"V_d{d}.csv")).values,
    }
    for d in dims
}

# ═══════════════ 2. FIGURE SETUP ═══════════════════════════════════════════
fig, axes = plt.subplots(1, 4, figsize=(10, 4), constrained_layout=True)

for ax, (tag, dat) in zip(axes, datasets.items()):
    X, V = dat["X"], dat["V"]

    # ── minimal AnnData
    adata = anndata.AnnData(X)
    adata.var_names = [f"g{i}" for i in range(X.shape[1])]
    adata.layers["X_raw"], adata.layers["V_raw"] = X, V

    # dummy colour column (constant) so we can pass "dummy" to color kwarg later if desired
    adata.obs["dummy"] = 1.0

    # ── UMAP WITHOUT PCA
    dyn.tl.reduceDimension(
        adata,
        X_data=X,
        reduction_method="umap",
        n_neighbors=30,
        min_dist=0.3,
        apply_pca=False,
        enforce=True,
    )

    # ── project velocities (again no PCA)
    dyn.tl.cell_velocities(
        adata,
        ekey="X_raw",
        vkey="V_raw",
        X=X,
        V=V,
        X_embedding=adata.obsm["X_umap"],
        basis="umap",
        method="pearson",
        apply_pca=False,
        enforce=True,
    )

    # Dynamo stores basis velocities as "<basis>_velocity" (here: "umap_velocity")
    V_umap = adata.obsm["velocity_umap"]
    X_umap = adata.obsm["X_umap"]

    # ════════════ 3. QUIVER PLOT (same aesthetics as earlier) ══════════════
    ax.scatter(
        X_umap[:, 0], X_umap[:, 1],
        c="#FFD700", s=150, alpha=0.10, edgecolors="none"
    )

    ax.quiver(
        X_umap[:, 0], X_umap[:, 1],
        V_umap[:, 0], V_umap[:, 1],
        color="black",
        angles="xy",
        scale_units="xy",
        scale=0.5,      # smaller → longer shafts
        width=0.004,
        headwidth=3,
        headlength=4,
        headaxislength=3,
    )

    ax.set_title(tag)
    ax.set_aspect("equal")
    ax.axis("off")

plt.show()
